# LeetCode 1504 — Count Submatrices With All Ones bằng Python

Link đề: https://leetcode.com/problems/count-submatrices-with-all-ones/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Stack, Matrix, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def numSubmat(self, mat: list[list[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biết cách **đếm** mọi hình chữ nhật con (submatrix) trong ma trận mà không bỏ sót / đếm trùng: mỗi hình được "neo" vào **một góc** (ở đây: góc dưới–phải).
2. Hiểu kỹ thuật **chiều cao cột** (histogram) `h[j]` — biến bài 2D thành nhiều bài 1D, giống bài *Maximal Rectangle*.
3. Viết được 4 cách, từ chậm đến nhanh: brute force + prefix sum 2D `O(m²n²)`, cố định cặp hàng `O(m²n)`, chiều cao + quét trái `O(m·n²)`, **stack đơn điệu** `O(m·n)`.
4. Nắm công thức đệ quy của stack đơn điệu: `cnt[j] = h[j]·(j − prev) + cnt[prev]` và vì sao nó đúng.
5. Tự kiểm chứng các cách bằng test ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `mat` kích thước `m × n` (mỗi ô là `0` hoặc `1`). Hãy đếm số **ma trận con** (hình chữ nhật gồm các hàng liên tiếp và các cột liên tiếp) mà **mọi ô đều bằng 1**.

**Ví dụ 1:** `mat = [[1,0,1],[1,1,0],[1,1,0]]` → `13`
(6 hình 1×1, 2 hình 1×2, 3 hình 2×1, 1 hình 2×2, 1 hình 3×1 — ở đây "a×b" là a hàng × b cột theo cách đề viết).

**Ví dụ 2:** `mat = [[0,1,1,0],[0,1,1,1],[1,1,1,0]]` → `24`

**Ràng buộc:** `1 ≤ m, n ≤ 150`, `mat[i][j] ∈ {0, 1}`.

Hai hình được coi là khác nhau nếu vị trí khác nhau (dù cùng kích thước).

In [1]:
import random
import time
from typing import List

examples = [
    ([[1, 0, 1], [1, 1, 0], [1, 1, 0]], 13),
    ([[0, 1, 1, 0], [0, 1, 1, 1], [1, 1, 1, 0]], 24),
]

def show(mat):
    for row in mat:
        print(" ".join("█" if x else "·" for x in row))

for mat, want in examples:
    show(mat)
    print("kỳ vọng:", want)
    print()

█ · █
█ █ ·
█ █ ·
kỳ vọng: 13

· █ █ ·
· █ █ █
█ █ █ ·
kỳ vọng: 24



## 3. Kiến thức nền

### 3.1 Duyệt ma trận 2 chiều

`mat` là list các list. `len(mat)` = số hàng `m`, `len(mat[0])` = số cột `n`. Ô hàng `i` cột `j` là `mat[i][j]`.

In [2]:
mat = [[1, 0, 1], [1, 1, 0]]
m, n = len(mat), len(mat[0])
print("m =", m, "n =", n)
for i in range(m):
    for j in range(n):
        print(f"mat[{i}][{j}]={mat[i][j]}", end="  ")
    print()

m = 2 n = 3
mat[0][0]=1  mat[0][1]=0  mat[0][2]=1  
mat[1][0]=1  mat[1][1]=1  mat[1][2]=0  


### 3.2 Tạo ma trận mới — bẫy `[[0]*n]*m`

`[[0]*n]*m` tạo **m tham chiếu tới cùng một list** — sửa một hàng là sửa tất cả. Phải dùng list comprehension.

In [3]:
bad = [[0] * 3] * 2
bad[0][0] = 9
print("bad :", bad)          # cả hai hàng đều đổi!

good = [[0] * 3 for _ in range(2)]
good[0][0] = 9
print("good:", good)

bad : [[9, 0, 0], [9, 0, 0]]
good: [[9, 0, 0], [0, 0, 0]]


### 3.3 Số đoạn con của một đoạn dài L

Một đoạn liên tiếp dài `L` có `L·(L+1)/2` đoạn con liên tiếp: chọn điểm đầu ≤ điểm cuối. Ví dụ `L = 3`: `[0],[1],[2],[0..1],[1..2],[0..2]` = 6.

In [4]:
def num_segments(L):
    return L * (L + 1) // 2

for L in range(6):
    brute = sum(1 for a in range(L) for b in range(a, L))
    print(L, num_segments(L), brute)

0 0 0
1 1 1
2 3 3
3 6 6
4 10 10
5 15 15


### 3.4 Prefix sum 2D

`P[i+1][j+1] = tổng các ô trong hình chữ nhật (0,0)–(i,j)`. Tổng hình `(r1,c1)–(r2,c2)`:

`P[r2+1][c2+1] − P[r1][c2+1] − P[r2+1][c1] + P[r1][c1]`.

Một hình toàn 1 ⇔ tổng = diện tích.

In [5]:
def prefix2d(mat):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = mat[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return P

def rect_sum(P, r1, c1, r2, c2):
    return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]

P = prefix2d(examples[0][0])
print(rect_sum(P, 1, 0, 2, 1), "(hình 2x2 ở góc dưới trái, diện tích 4)")
print(rect_sum(P, 0, 0, 2, 2), "(cả ma trận)")

4 (hình 2x2 ở góc dưới trái, diện tích 4)
6 (cả ma trận)


### 3.5 Stack (ngăn xếp) bằng list

`append` để đẩy, `pop()` để lấy phần tử trên cùng, `st[-1]` để xem đỉnh. Stack **đơn điệu** là stack mà các phần tử luôn tăng (hoặc giảm) từ đáy lên đỉnh: trước khi đẩy `x`, ta `pop` mọi phần tử vi phạm.

In [6]:
arr = [3, 1, 4, 1, 5, 2]
st = []                       # lưu chỉ số, giá trị tăng nghiêm ngặt
prev_smaller = []
for j, x in enumerate(arr):
    while st and arr[st[-1]] >= x:
        st.pop()
    prev_smaller.append(st[-1] if st else -1)
    st.append(j)
print("arr          :", arr)
print("prev nhỏ hơn :", prev_smaller)

arr          : [3, 1, 4, 1, 5, 2]
prev nhỏ hơn : [-1, -1, 1, -1, 3, 3]


## 4. Ý tưởng nền tảng / trực giác

**Neo mỗi hình vào góc dưới–phải.** Mỗi hình chữ nhật có đúng một góc dưới–phải `(i, j)`. Vậy:

`đáp án = Σ_{(i,j)} (số hình toàn 1 có góc dưới–phải tại (i, j))`.

**Chiều cao cột.** Với hàng `i`, đặt `h[j]` = số ô 1 liên tiếp **kéo lên** từ `(i, j)` (gặp 0 thì dừng). Một hình có góc dưới–phải `(i, j)`, cột trái `k`, cao `t` hàng là toàn 1 ⇔ `t ≤ min(h[k..j])`. Vậy số hình có góc dưới–phải `(i,j)` là:

`Σ_{k = j, j−1, …} min(h[k..j])` (dừng khi min = 0).

Tính tay Ví dụ 1, hàng cuối `i = 2`: `mat[2] = [1,1,0]`, chiều cao `h = [3, 2, 0]`.
- `j = 0`: chỉ `k = 0` → `min = 3` → 3 hình (cao 1, 2, 3).
- `j = 1`: `k = 1` → 2; `k = 0` → `min(3,2) = 2` → tổng 4.
- `j = 2`: `h = 0` → 0.
Hàng 2 đóng góp 7. Tương tự hàng 0 (`h=[1,0,1]`) đóng góp 2, hàng 1 (`h=[2,1,0]`) đóng góp `2 + (1+1) = 4`. Tổng `2 + 4 + 7 = 13` ✔.

In [7]:
mat = examples[0][0]
n = len(mat[0])
h = [0] * n
total = 0
for i, row in enumerate(mat):
    h = [h[j] + 1 if row[j] else 0 for j in range(n)]
    contrib = []
    for j in range(n):
        s, mn = 0, float("inf")
        for k in range(j, -1, -1):
            mn = min(mn, h[k])
            if mn == 0:
                break
            s += mn
        contrib.append(s)
    total += sum(contrib)
    print(f"hàng {i}: h={h} đóng góp theo cột={contrib} tổng hàng={sum(contrib)}")
print("tổng:", total)

hàng 0: h=[1, 0, 1] đóng góp theo cột=[1, 0, 1] tổng hàng=2
hàng 1: h=[2, 1, 0] đóng góp theo cột=[2, 2, 0] tổng hàng=4
hàng 2: h=[3, 2, 0] đóng góp theo cột=[3, 4, 0] tổng hàng=7
tổng: 13


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** liệt kê mọi bộ `(r1, c1, r2, c2)` với `r1 ≤ r2`, `c1 ≤ c2`; kiểm tra hình đó có toàn 1 không.

- Kiểm tra bằng vòng lặp trực tiếp: `O(m²n² · mn)` — chỉ dùng cho ma trận rất nhỏ.
- Kiểm tra bằng **prefix sum 2D**: `O(1)` mỗi hình → tổng `O(m²n²)`. Với `m = n = 150` là `≈ 1.3·10⁸` bộ — vẫn quá chậm trong Python, nhưng là **đáp án tham chiếu** tuyệt vời để kiểm tra các cách nhanh.

In [8]:
def brute_naive(mat):
    m, n = len(mat), len(mat[0])
    cnt = 0
    for r1 in range(m):
        for r2 in range(r1, m):
            for c1 in range(n):
                for c2 in range(c1, n):
                    if all(mat[r][c] for r in range(r1, r2 + 1) for c in range(c1, c2 + 1)):
                        cnt += 1
    return cnt

def brute_prefix(mat):
    m, n = len(mat), len(mat[0])
    P = prefix2d(mat)
    cnt = 0
    for r1 in range(m):
        for r2 in range(r1, m):
            for c1 in range(n):
                for c2 in range(c1, n):
                    area = (r2 - r1 + 1) * (c2 - c1 + 1)
                    if rect_sum(P, r1, c1, r2, c2) == area:
                        cnt += 1
    return cnt

for mat, want in examples:
    print(brute_naive(mat), brute_prefix(mat), "kỳ vọng", want)

13 13 kỳ vọng 13
24 24 kỳ vọng 24


**Độ phức tạp:** `brute_prefix` — Time `O(m²n²)`, Space `O(mn)` cho bảng prefix.

**Nút thắt:** ta kiểm tra **từng** hình một, kể cả khi đã biết chắc nó chứa 0. Ví dụ nếu hình `(r1,c1)–(r2,c2)` chứa 0 thì mọi hình mở rộng sang phải cũng chứa 0 — có thể `break` sớm. Cải tiến tự nhiên: cố định một chiều, xử lý chiều còn lại bằng "đếm đoạn".

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (cố định cặp hàng):** cố định hàng trên `r1` và hàng dưới `r2`. Gộp các hàng này thành một hàng `col_ok[c] = 1` nếu cột `c` toàn 1 trên mọi hàng `r1..r2`. Số hình có đúng các hàng `r1..r2` = số đoạn con toàn 1 của `col_ok` = `Σ L(L+1)/2` trên các đoạn 1 liên tiếp. Khi tăng `r2`, cập nhật `col_ok[c] &= mat[r2][c]` trong `O(n)`.

**Insight 2 (chiều cao):** như mục 4, neo góc dưới–phải và dùng `h[j]`.

**Insight 3 (stack đơn điệu):** gọi `cnt[j]` = số hình có góc dưới–phải `(i, j)`. Gọi `p` = chỉ số gần nhất bên trái có `h[p] < h[j]`.
- Với cột trái `k ∈ (p, j]`: `min(h[k..j]) = h[j]` → mỗi `k` cho `h[j]` hình → tổng `h[j]·(j − p)`.
- Với cột trái `k ≤ p`: `min(h[k..j]) = min(h[k..p])` (vì mọi cột trong `(p, j]` đều ≥ `h[j]` > `h[p]`) → đúng bằng `cnt[p]`.

**Quy tắc:** *nếu* `h[st[-1]] ≥ h[j]` *thì* pop (cột đó không bao giờ là "prev nhỏ hơn" cho cột nào bên phải nữa); sau đó `cnt[j] = h[j]·(j − p) + cnt[p]`.

## 7. Các cách tiếp theo

### Cách 2 — Cố định cặp hàng + đếm cột liên tiếp `O(m²·n)`

Từng bước:
1. Với mỗi `r1`: khởi tạo `ok = [1]*n`.
2. Với mỗi `r2 ≥ r1`: `ok[c] &= mat[r2][c]`.
3. Quét `ok`, giữ `run` = độ dài đoạn 1 đang kéo dài đến cột `c`; mỗi cột cộng `run` vào kết quả (số đoạn kết thúc tại `c` đúng bằng `run` — tương đương `L(L+1)/2` khi cộng dồn).

In [9]:
def fixed_rows(mat):
    m, n = len(mat), len(mat[0])
    ans = 0
    for r1 in range(m):
        ok = [1] * n
        for r2 in range(r1, m):
            run = 0
            for c in range(n):
                ok[c] &= mat[r2][c]
                run = run + 1 if ok[c] else 0
                ans += run
    return ans

for mat, want in examples:
    print(fixed_rows(mat), "kỳ vọng", want)

13 kỳ vọng 13
24 kỳ vọng 24


Bản nộp LeetCode của Cách 2 (≈ `150²·150/2 ≈ 1.7·10⁶` phép — chạy tốt):

In [10]:
class SolutionFixedRows:
    def numSubmat(self, mat: list[list[int]]) -> int:
        m, n = len(mat), len(mat[0])
        ans = 0
        for r1 in range(m):
            ok = [1] * n
            for r2 in range(r1, m):
                run = 0
                row = mat[r2]
                for c in range(n):
                    ok[c] &= row[c]
                    run = run + 1 if ok[c] else 0
                    ans += run
        return ans

print([SolutionFixedRows().numSubmat(m) for m, _ in examples])

[13, 24]


**Độ phức tạp:** Time `O(m²·n)`, Space `O(n)`. Mẹo: nếu `m > n` có thể chuyển vị ma trận để được `O(min(m,n)²·max(m,n))`.

### Cách 3 — Chiều cao cột + quét sang trái `O(m·n²)`

Đây là cách gợi ý của LeetCode (hint 1, 2):
1. Mỗi hàng cập nhật `h[j] = h[j] + 1` nếu `mat[i][j] == 1`, ngược lại `0`.
2. Với mỗi `j`, đi sang trái `k = j, j−1, …`, giữ `mn = min(h[k..j])`, cộng `mn`; dừng khi `mn == 0`.

In [11]:
def heights_scan(mat):
    m, n = len(mat), len(mat[0])
    h = [0] * n
    ans = 0
    for i in range(m):
        for j in range(n):
            h[j] = h[j] + 1 if mat[i][j] else 0
        for j in range(n):
            mn = h[j]
            k = j
            while k >= 0 and mn > 0:
                mn = min(mn, h[k])
                ans += mn
                k -= 1
    return ans

for mat, want in examples:
    print(heights_scan(mat), "kỳ vọng", want)

13 kỳ vọng 13
24 kỳ vọng 24


Chú ý thứ tự trong vòng `while`: cập nhật `mn` bằng `h[k]` **trước** khi cộng. Nếu `h[k] = 0` thì cộng 0 rồi vòng lặp dừng ở lượt sau — không sai.

In [12]:
class SolutionHeights:
    def numSubmat(self, mat: list[list[int]]) -> int:
        m, n = len(mat), len(mat[0])
        h = [0] * n
        ans = 0
        for i in range(m):
            row = mat[i]
            for j in range(n):
                h[j] = h[j] + 1 if row[j] else 0
            for j in range(n):
                mn = h[j]
                k = j
                while k >= 0 and mn > 0:
                    if h[k] < mn:
                        mn = h[k]
                    ans += mn
                    k -= 1
        return ans

print([SolutionHeights().numSubmat(m) for m, _ in examples])

[13, 24]


**Độ phức tạp:** Time `O(m·n²)` worst-case (ma trận toàn 1), Space `O(n)`.

### Cách 4 — Stack đơn điệu `O(m·n)`

Từng đoạn:
1. Cập nhật `h` như Cách 3.
2. Duyệt `j` từ trái sang phải với stack chỉ số có `h` **tăng nghiêm ngặt**.
3. Pop mọi chỉ số có `h ≥ h[j]`. Đỉnh còn lại (nếu có) là `p`, nếu rỗng `p = −1`.
4. `cnt[j] = h[j]·(j − p) + (cnt[p] if p ≥ 0 else 0)`; cộng `cnt[j]` vào đáp án; đẩy `j`.

In [13]:
def mono_stack(mat):
    m, n = len(mat), len(mat[0])
    h = [0] * n
    ans = 0
    for i in range(m):
        for j in range(n):
            h[j] = h[j] + 1 if mat[i][j] else 0
        cnt = [0] * n
        st = []
        for j in range(n):
            while st and h[st[-1]] >= h[j]:
                st.pop()
            p = st[-1] if st else -1
            cnt[j] = h[j] * (j - p) + (cnt[p] if p >= 0 else 0)
            ans += cnt[j]
            st.append(j)
    return ans

for mat, want in examples:
    print(mono_stack(mat), "kỳ vọng", want)

13 kỳ vọng 13
24 kỳ vọng 24


In [14]:
class SolutionStack:
    def numSubmat(self, mat: list[list[int]]) -> int:
        n = len(mat[0])
        h = [0] * n
        ans = 0
        for row in mat:
            for j in range(n):
                h[j] = h[j] + 1 if row[j] else 0
            cnt = [0] * n
            st = []
            for j in range(n):
                while st and h[st[-1]] >= h[j]:
                    st.pop()
                if st:
                    p = st[-1]
                    cnt[j] = h[j] * (j - p) + cnt[p]
                else:
                    cnt[j] = h[j] * (j + 1)
                ans += cnt[j]
                st.append(j)
        return ans

print([SolutionStack().numSubmat(m) for m, _ in examples])

[13, 24]


**Độ phức tạp:** mỗi chỉ số được đẩy và pop tối đa 1 lần mỗi hàng → Time `O(m·n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 1. Bảng trạng thái (hàng 2, `h = [3, 2, 0]`):

| j | h[j] | pop | p | cnt[j] = h·(j−p) + cnt[p] | stack sau |
|---|---|---|---|---|---|
| 0 | 3 | — | −1 | 3·1 + 0 = 3 | [0] |
| 1 | 2 | 0 (h=3 ≥ 2) | −1 | 2·2 + 0 = 4 | [1] |
| 2 | 0 | 1 (h=2 ≥ 0) | −1 | 0·3 + 0 = 0 | [2] |

Hàng 2 đóng góp `3 + 4 + 0 = 7`, khớp mục 4.

In [15]:
def trace_stack(mat):
    n = len(mat[0])
    h = [0] * n
    ans = 0
    for i, row in enumerate(mat):
        h = [h[j] + 1 if row[j] else 0 for j in range(n)]
        print(f"=== hàng {i}: row={row} h={h}")
        cnt = [0] * n
        st = []
        for j in range(n):
            popped = []
            while st and h[st[-1]] >= h[j]:
                popped.append(st.pop())
            p = st[-1] if st else -1
            cnt[j] = h[j] * (j - p) + (cnt[p] if p >= 0 else 0)
            ans += cnt[j]
            st.append(j)
            print(f"  j={j} h={h[j]} pop={popped} p={p} cnt={cnt[j]} stack={st} ans={ans}")
    return ans

print("kết quả:", trace_stack(examples[0][0]))

=== hàng 0: row=[1, 0, 1] h=[1, 0, 1]
  j=0 h=1 pop=[] p=-1 cnt=1 stack=[0] ans=1
  j=1 h=0 pop=[0] p=-1 cnt=0 stack=[1] ans=1
  j=2 h=1 pop=[] p=1 cnt=1 stack=[1, 2] ans=2
=== hàng 1: row=[1, 1, 0] h=[2, 1, 0]
  j=0 h=2 pop=[] p=-1 cnt=2 stack=[0] ans=4
  j=1 h=1 pop=[0] p=-1 cnt=2 stack=[1] ans=6
  j=2 h=0 pop=[1] p=-1 cnt=0 stack=[2] ans=6
=== hàng 2: row=[1, 1, 0] h=[3, 2, 0]
  j=0 h=3 pop=[] p=-1 cnt=3 stack=[0] ans=9
  j=1 h=2 pop=[0] p=-1 cnt=4 stack=[1] ans=13
  j=2 h=0 pop=[1] p=-1 cnt=0 stack=[2] ans=13
kết quả: 13


Trace Cách 2 (cặp hàng) trên Ví dụ 1 — in `ok` và số hình đếm được cho mỗi cặp `(r1, r2)`:

In [16]:
def trace_fixed_rows(mat):
    m, n = len(mat), len(mat[0])
    ans = 0
    for r1 in range(m):
        ok = [1] * n
        for r2 in range(r1, m):
            run, add = 0, 0
            for c in range(n):
                ok[c] &= mat[r2][c]
                run = run + 1 if ok[c] else 0
                add += run
            ans += add
            print(f"r1={r1} r2={r2} ok={ok} +{add} -> ans={ans}")
    return ans

print("kết quả:", trace_fixed_rows(examples[0][0]))

r1=0 r2=0 ok=[1, 0, 1] +2 -> ans=2
r1=0 r2=1 ok=[1, 0, 0] +1 -> ans=3
r1=0 r2=2 ok=[1, 0, 0] +1 -> ans=4
r1=1 r2=1 ok=[1, 1, 0] +3 -> ans=7
r1=1 r2=2 ok=[1, 1, 0] +3 -> ans=10
r1=2 r2=2 ok=[1, 1, 0] +3 -> ans=13
kết quả: 13


## 9. Code nộp LeetCode

Bản chính: stack đơn điệu `O(m·n)`.

In [17]:
class Solution:
    def numSubmat(self, mat: list[list[int]]) -> int:
        n = len(mat[0])
        h = [0] * n
        ans = 0
        for row in mat:
            for j in range(n):
                h[j] = h[j] + 1 if row[j] else 0
            cnt = [0] * n
            st = []
            for j in range(n):
                while st and h[st[-1]] >= h[j]:
                    st.pop()
                if st:
                    p = st[-1]
                    cnt[j] = h[j] * (j - p) + cnt[p]
                else:
                    cnt[j] = h[j] * (j + 1)
                ans += cnt[j]
                st.append(j)
        return ans

for mat, want in examples:
    assert Solution().numSubmat(mat) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Không trùng, không sót:** mỗi hình chữ nhật toàn 1 có duy nhất một góc dưới–phải `(i, j)`, một cột trái `k ≤ j` và một chiều cao `t ≥ 1`. Nó toàn 1 ⇔ với mọi cột `c ∈ [k, j]`, `t` ô từ hàng `i` lên đều là 1 ⇔ `t ≤ h[c]` ⇔ `t ≤ min(h[k..j])`. Vậy với `(i, j, k)` cố định có đúng `min(h[k..j])` hình → Cách 3 đúng.

**Bất biến của stack (Cách 4):** sau khi xử lý `j`, stack chứa các chỉ số có `h` tăng nghiêm ngặt, và mỗi phần tử là "chỉ số gần nhất bên trái có `h` nhỏ hơn" của phần tử ngay trên nó. Khi xử lý `j`, các chỉ số bị pop có `h ≥ h[j]`, nên với mọi `j' > j` chúng không thể là "prev nhỏ hơn" (vì `j` gần hơn và nhỏ hơn hoặc bằng). Do đó `p` tìm được đúng là chỉ số gần nhất bên trái có `h[p] < h[j]`.

**Công thức `cnt[j]`:** tách tổng `Σ_k min(h[k..j])` tại `p`: phần `k ∈ (p, j]` mỗi số hạng bằng `h[j]`; phần `k ≤ p` bằng `Σ_k min(h[k..p]) = cnt[p]` vì thêm các cột `(p, j]` (đều ≥ `h[j] > h[p]`) không làm giảm min. ∎

**Cách 2:** với cặp hàng cố định, hình hợp lệ ⇔ đoạn cột toàn 1 trên `ok`; số đoạn con kết thúc tại `c` bằng độ dài đoạn 1 đang chạy `run`.

## 11. Phân tích Time / Space

Gọi `m` = số hàng, `n` = số cột (`≤ 150`).

| Cách | Time | Space | Với 150×150 |
|---|---|---|---|
| Brute + prefix 2D | `O(m²n²)` | `O(mn)` | ~1.3·10⁸ — quá chậm (Python) |
| Cặp hàng | `O(m²n)` | `O(n)` | ~1.7·10⁶ |
| Chiều cao + quét trái | `O(mn²)` worst | `O(n)` | ~1.7·10⁶ worst (ma trận toàn 1) |
| Stack đơn điệu | `O(mn)` | `O(n)` | ~2.3·10⁴ |

Với Cách 3, average nhanh hơn worst nhiều vì vòng `while` dừng ngay khi gặp cột chiều cao 0.

## 12. Bộ test

Gồm: ví dụ đề, edge case (1×1, toàn 0, toàn 1, một hàng, một cột), đối chiếu ngẫu nhiên với brute force (seed cố định) và test thời gian 150×150.

In [18]:
ALL = {
    "brute_prefix": brute_prefix,
    "fixed_rows": lambda a: SolutionFixedRows().numSubmat(a),
    "heights": lambda a: SolutionHeights().numSubmat(a),
    "stack": lambda a: Solution().numSubmat(a),
}

def full_ones(m, n):
    return (m * (m + 1) // 2) * (n * (n + 1) // 2)

edge = [
    ([[0]], 0), ([[1]], 1),
    ([[0, 0], [0, 0]], 0),
    ([[1, 1], [1, 1]], 9),
    ([[1, 1, 1, 1]], 10),
    ([[1], [1], [1]], 6),
    ([[1, 0, 1, 0, 1]], 3),
    ([[1] * 4 for _ in range(3)], full_ones(3, 4)),
]
for mat, want in examples + edge:
    for name, f in ALL.items():
        got = f(mat)
        assert got == want, (name, mat, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [19]:
rng = random.Random(1504)
for _ in range(400):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    p = rng.choice([0.3, 0.6, 0.85, 1.0])
    mat = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]
    ref = brute_naive(mat)
    for name, f in ALL.items():
        assert f(mat) == ref, (name, mat)
print("400 ma trận ngẫu nhiên: mọi cách khớp brute force")

400 ma trận ngẫu nhiên: mọi cách khớp brute force


In [20]:
big_ones = [[1] * 150 for _ in range(150)]
rng2 = random.Random(7)
big_rand = [[1 if rng2.random() < 0.9 else 0 for _ in range(150)] for _ in range(150)]
for name in ["fixed_rows", "heights", "stack"]:
    t = time.perf_counter()
    r1 = ALL[name](big_ones)
    r2 = ALL[name](big_rand)
    print(f"{name:10s} toàn 1: {r1}  ngẫu nhiên: {r2}  ({time.perf_counter() - t:.2f}s)")
assert ALL["stack"](big_ones) == full_ones(150, 150)

fixed_rows toàn 1: 128255625  ngẫu nhiên: 563724  (0.34s)
heights    toàn 1: 128255625  ngẫu nhiên: 563724  (0.18s)
stack      toàn 1: 128255625  ngẫu nhiên: 563724  (0.01s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đặt lại chiều cao về 0 khi gặp ô 0

Viết `h[j] += mat[i][j]` thì chiều cao "nhảy qua" ô 0, coi cột như liền mạch.

In [21]:
def wrong_height(mat):
    n = len(mat[0])
    h = [0] * n
    ans = 0
    for row in mat:
        for j in range(n):
            h[j] += row[j]                       # SAI: không reset khi row[j] == 0
        for j in range(n):
            mn, k = h[j], j
            while k >= 0 and mn > 0:
                mn = min(mn, h[k]); ans += mn; k -= 1
    return ans

bad = [[1], [0], [1]]
print("sai :", wrong_height(bad), "| đúng:", Solution().numSubmat(bad))

sai : 4 | đúng: 2


**Sửa:** `h[j] = h[j] + 1 if row[j] else 0`.

### Lỗi 2 — Stack đơn điệu quên cộng `cnt[p]`

Chỉ tính `h[j]·(j − p)` là bỏ mất các hình có cột trái ở bên trái `p` (chiều cao bị giới hạn bởi cột thấp hơn).

In [22]:
def wrong_stack(mat):
    n = len(mat[0])
    h = [0] * n
    ans = 0
    for row in mat:
        h = [h[j] + 1 if row[j] else 0 for j in range(n)]
        st = []
        for j in range(n):
            while st and h[st[-1]] >= h[j]:
                st.pop()
            p = st[-1] if st else -1
            ans += h[j] * (j - p)                # SAI: thiếu cnt[p]
            st.append(j)
    return ans

bad = [[0, 1], [1, 1]]
print("sai :", wrong_stack(bad), "| đúng:", Solution().numSubmat(bad))

sai : 4 | đúng: 5


Ở đây hàng 1 có `h = [1, 2]`; với `j = 1`, `p = 0` nên bản sai chỉ cộng `2·1 = 2`, quên hình `1×2` ở hàng dưới (đóng góp `cnt[0] = 1`).

### Lỗi 3 — Tạo ma trận phụ bằng `[[0]*n]*m`

Ví dụ muốn lưu chiều cao của mọi hàng trong `H[i][j]` — nhưng các hàng dùng chung một list.

In [23]:
def wrong_alias(mat):
    m, n = len(mat), len(mat[0])
    H = [[0] * n] * m                            # SAI: m tham chiếu cùng 1 list
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                H[i][j] = (H[i - 1][j] if i else 0) + 1
            else:
                H[i][j] = 0
    ans = 0
    for i in range(m):
        for j in range(n):
            mn, k = H[i][j], j
            while k >= 0 and mn > 0:
                mn = min(mn, H[i][k]); ans += mn; k -= 1
    return ans

bad = examples[0][0]
print("sai :", wrong_alias(bad), "| đúng:", Solution().numSubmat(bad))

sai : 21 | đúng: 13


**Sửa:** `H = [[0] * n for _ in range(m)]`.

### Lỗi 4 — Quét trái dùng `h[k]` thay vì min tích luỹ

Cộng `h[k]` cho mỗi cột trái là sai: chiều cao hình bị giới hạn bởi cột **thấp nhất** trong đoạn, không phải cột trái.

In [24]:
def wrong_min(mat):
    n = len(mat[0])
    h = [0] * n
    ans = 0
    for row in mat:
        h = [h[j] + 1 if row[j] else 0 for j in range(n)]
        for j in range(n):
            k = j
            while k >= 0 and h[k] > 0:
                ans += h[k]                      # SAI: phải là min(h[k..j])
                k -= 1
    return ans

bad = [[1, 0], [1, 1]]
print("sai :", wrong_min(bad), "| đúng:", Solution().numSubmat(bad))

sai : 6 | đúng: 5


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm/tìm hình chữ nhật toàn 1 trong ma trận nhị phân"** → nghĩ ngay tới **chiều cao cột** `h[j]` để chuyển về bài histogram 1D (LC 84, 85, 1504, 1277).
- **"Đếm mọi đoạn con, mỗi đoạn đóng góp min"** → *sum of subarray minimums* (LC 907) → stack đơn điệu + "prev nhỏ hơn".
- **Neo vào một góc / một đầu mút** là cách chuẩn để đếm không trùng.
- Ràng buộc `≤ 150` cho phép `O(n³) ≈ 3.4·10⁶` — trong phỏng vấn có thể trình bày Cách 3 trước, rồi nâng lên Cách 4.
- Luôn giữ một hàm brute force nhỏ để đối chiếu.

## 15. Biến thể của bài

1. **LC 1277 – Count Square Submatrices with All Ones:** chỉ đếm hình vuông → DP `dp[i][j] = 1 + min(trên, trái, chéo)`.
2. **LC 85 – Maximal Rectangle:** tìm diện tích lớn nhất thay vì đếm → histogram + stack.
3. **LC 221 – Maximal Square.**
4. **LC 907 – Sum of Subarray Minimums:** chính là bài 1D bên trong Cách 4.
5. Đếm hình chữ nhật toàn 0 → đổi vai 0/1.
6. Đếm hình có **tổng = target** (LC 1074) → cố định cặp hàng + prefix sum + hash map (giống Cách 2).

In [25]:
# Minh hoạ biến thể: đếm hình chữ nhật toàn 0 = áp dụng lời giải lên ma trận đảo bit
def count_all_zero(mat):
    inv = [[1 - x for x in row] for row in mat]
    return Solution().numSubmat(inv)

print(count_all_zero(examples[0][0]), brute_naive([[1 - x for x in r] for r in examples[0][0]]))

4 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute + prefix 2D | Liệt kê mọi hình, kiểm tra tổng = diện tích | `O(m²n²)` | `O(mn)` | Đối chiếu, ma trận nhỏ |
| 2. Cặp hàng | Gộp hàng `r1..r2` bằng AND, đếm đoạn 1 | `O(m²n)` | `O(n)` | Dễ nghĩ, đủ nhanh với 150 |
| 3. Chiều cao + quét trái | Neo góc dưới–phải, cộng min tích luỹ | `O(mn²)` | `O(n)` | Lời giải gợi ý, dễ viết |
| 4. Stack đơn điệu | `cnt[j] = h[j]·(j−p) + cnt[p]` | `O(mn)` | `O(n)` | Tối ưu, phỏng vấn nâng cao |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().numSubmat([[1, 1, 1]])` và `Solution().numSubmat([[1, 1], [0, 1]])` trả về bao nhiêu?

### Bài 2 (tính tay)
Một ma trận `m × n` toàn 1 có bao nhiêu ma trận con? Tính cho `2 × 3` và tìm công thức tổng quát.

### Bài 3 (điền TODO)
Viết `count_row(row)`: đếm số đoạn con toàn 1 của một list 0/1 (bài 1D, `O(n)`).

### Bài 4 (tự viết lại)
Viết lại Cách 3 nhưng **chuyển vị** ma trận nếu `m < n` để vòng lặp bậc hai chạy trên chiều ngắn hơn: `numSubmat_transpose(mat)`.

### Bài 5 (mini challenge)
LC 1277: đếm số **hình vuông** con toàn 1: `count_squares(mat)`.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_row(row):
    # TODO: giữ run = độ dài đoạn 1 kết thúc tại vị trí hiện tại, cộng dồn run
    pass

# Bài 4 — TODO
def numSubmat_transpose(mat):
    # TODO: nếu số hàng < số cột thì mat = list(map(list, zip(*mat))), rồi chạy Cách 3
    pass

# Bài 5 — TODO
def count_squares(mat):
    # TODO: dp[i][j] = cạnh hình vuông lớn nhất có góc dưới-phải (i, j)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một hàng dài 3 → số đoạn con. (2) Ma trận thứ hai: đếm 1×1, rồi các hình lớn hơn. (3) `6` và … dùng `h` của hàng 1 = `[0, 2]`.

**Bài 2.** (1) Một hình được xác định bởi đoạn hàng và đoạn cột độc lập. (2) Số đoạn hàng = `m(m+1)/2`. (3) Nhân với số đoạn cột `n(n+1)/2`.

**Bài 3.** (1) Số đoạn kết thúc tại `j` bằng độ dài đoạn 1 kết thúc tại `j`. (2) Gặp 0 thì `run = 0`. (3) `ans += run` mỗi bước.

**Bài 4.** (1) Đáp án không đổi khi chuyển vị. (2) `zip(*mat)` trả các tuple cột. (3) Sau khi chuyển vị, `n` mới là chiều ngắn hơn → `O(max·min²)`.

**Bài 5.** (1) Ô 0 → `dp = 0`. (2) Ô 1 → `1 + min(dp trên, dp trái, dp chéo trên-trái)`. (3) Đáp án = tổng mọi `dp[i][j]` (hình vuông cạnh `k` góc dưới–phải ở đó với mọi `k ≤ dp`).

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
print(Solution().numSubmat([[1, 1, 1]]), Solution().numSubmat([[1, 1], [0, 1]]))
assert Solution().numSubmat([[1, 1, 1]]) == 6
assert Solution().numSubmat([[1, 1], [0, 1]]) == 5   # 3 ô + 1 ngang (hàng 0) + 1 dọc (cột 1)

# Bài 2
def full_count(m, n):
    return (m * (m + 1) // 2) * (n * (n + 1) // 2)
assert full_count(2, 3) == 18 == brute_naive([[1] * 3 for _ in range(2)])
print("Bài 1, 2: OK")

6 5
Bài 1, 2: OK


In [28]:
# Bài 3
def count_row_sol(row):
    ans = run = 0
    for x in row:
        run = run + 1 if x else 0
        ans += run
    return ans

assert count_row_sol([1, 1, 0, 1, 1, 1]) == 3 + 6
assert all(count_row_sol(r[0]) == brute_naive(r) for r in [[[1, 0, 1]], [[1, 1, 1, 1]], [[0]]])

# Bài 4
def numSubmat_transpose_sol(mat):
    if len(mat) < len(mat[0]):
        mat = [list(col) for col in zip(*mat)]
    return SolutionHeights().numSubmat(mat)

rng = random.Random(42)
for _ in range(200):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    mat = [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)]
    assert numSubmat_transpose_sol(mat) == Solution().numSubmat(mat)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [29]:
# Bài 5
def count_squares_sol(mat):
    m, n = len(mat), len(mat[0])
    dp = [[0] * n for _ in range(m)]
    total = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                if i and j:
                    dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
                else:
                    dp[i][j] = 1
                total += dp[i][j]
    return total

def brute_squares(mat):
    m, n = len(mat), len(mat[0])
    c = 0
    for i in range(m):
        for j in range(n):
            for k in range(1, min(m - i, n - j) + 1):
                if all(mat[r][s] for r in range(i, i + k) for s in range(j, j + k)):
                    c += 1
    return c

assert count_squares_sol([[0, 1, 1, 1], [1, 1, 1, 1], [0, 1, 1, 1]]) == 15   # ví dụ LC 1277
for _ in range(200):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    mat = [[1 if rng.random() < 0.75 else 0 for _ in range(n)] for _ in range(m)]
    assert count_squares_sol(mat) == brute_squares(mat)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "histogram + stack đơn điệu đếm tổng min":**

```python
h = [0] * n
for row in mat:
    for j in range(n):
        h[j] = h[j] + 1 if row[j] else 0       # reset khi gặp 0
    cnt, st = [0] * n, []
    for j in range(n):
        while st and h[st[-1]] >= h[j]:
            st.pop()
        p = st[-1] if st else -1
        cnt[j] = h[j] * (j - p) + (cnt[p] if p >= 0 else 0)
        ans += cnt[j]
        st.append(j)
```

**Checklist trước khi submit:**
- [ ] Chiều cao reset về 0 khi `row[j] == 0`.
- [ ] Đếm theo **góc dưới–phải** để không trùng.
- [ ] Stack: điều kiện pop `>=` và nhớ cộng `cnt[p]`.
- [ ] Không tạo ma trận bằng `[[0]*n]*m`.
- [ ] Đã đối chiếu với brute force trên ma trận nhỏ ngẫu nhiên.
- [ ] Đúng chữ ký: `numSubmat(self, mat: list[list[int]]) -> int`.